# 06 — Do arquivo ao céu variável: alertas em tempo real

Workshop SAB — AI-Scope / ADSS

**Pergunta:** como organizar uma decisão científica quando os eventos chegam em fluxo? Aqui usamos um replay finito para que a demonstração termine sozinha. As categorias e os campos do payload são descobertos na instância atual.

In [ ]:
import json
import os
from datetime import datetime, timezone

import adss
import pandas as pd

username = os.getenv("AI_SCOPE_USERNAME")
password = os.getenv("AI_SCOPE_PASSWORD")
if not username or not password:
    raise RuntimeError("Defina AI_SCOPE_USERNAME e AI_SCOPE_PASSWORD antes de executar.")
cl = adss.ADSSClient(base_url="https://ai-scope.cbpf.br/", username=username, password=password)

## Categorias disponíveis

Categorias podem mudar entre instâncias. O replay abaixo usa `all` e limita a 20 alertas; `follow=False` encerra após os eventos retidos. Em produção, um consumidor contínuo precisa de política de reconexão, deduplicação e checkpoint.

In [ ]:
categories = pd.DataFrame(cl.get_alert_categories())
display(categories)

alerts = list(cl.listen_alerts(categories="all", replay="earliest", limit=20, follow=False))
print(f"Alertas recebidos: {len(alerts)}")
if alerts:
    print("Campos do primeiro alerta:", sorted(alerts[0]))
    print(json.dumps(alerts[0], ensure_ascii=False, indent=2, default=str)[:3000])
else:
    print("Sem alertas retidos no momento; os exercícios abaixo continuam conceituais.")

## Inventário antes de priorizar

Não presumimos que todo evento tem magnitude, classe ou coordenadas. Primeiro medimos a presença dos campos. Só depois vale escrever uma regra dependente do esquema real. Um ID de evento e o horário de recebimento permitem rastrear decisões e evitar processar o mesmo replay duas vezes.

In [ ]:
if alerts:
    inventory = pd.DataFrame([
        {"field": field, "present": sum(isinstance(a, dict) and a.get(field) is not None for a in alerts)}
        for field in sorted(set().union(*(a.keys() for a in alerts if isinstance(a, dict))))
    ]).sort_values(["present", "field"], ascending=[False, True])
    display(inventory)
else:
    inventory = pd.DataFrame(columns=["field", "present"])

received_utc = datetime.now(timezone.utc).isoformat()
print("Replay analisado em", received_utc)

## Exercício de triagem

1. Escolha do inventário um identificador estável, posição, instante e sinais de confiança. Que campos estão ausentes?
2. Defina uma regra de prioridade com critérios explícitos e uma saída `review`, `follow_up` ou `ignore`. Qual critério exige consulta adicional ao AI-Scope?
3. Enriqueça poucos candidatos com `cone()` em Gaia ou com cutouts, como nos notebooks 04 e 05. Registre raio, tabela, versão da regra/modelo e horário da decisão.
4. Repita o replay: sua regra produz a mesma decisão por ID? Descreva como evitar duplicatas e como reavaliar um evento quando chega nova informação.

O replay não reproduz latência nem taxa real de chegada; meça esses fatores antes de implementar um consumidor contínuo.